# Experiment: Huemiliator Eval Surface

Objective:
- inspect the local SQLite evidence lane for deterministic one-up outputs
- confirm that the CLI and notebook read the same stored fields and verdict state

Success criteria:
- the local DB initializes cleanly
- rows can be logged, judged, and inspected
- family counts and pending rows are easy to review


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from huemiliator.config import load_settings
from huemiliator.eval_db import connect, counts, init_db, list_outputs, record_one_up_state
from huemiliator.pipeline import build_one_up_state
from huemiliator.swatches import load_swatch_snapshot

settings = load_settings()
dataset = load_swatch_snapshot(settings.swatch_snapshot_path)
db_path = init_db(settings.eval_db_path)

{
    "repo_root": str(REPO_ROOT),
    "db_path": str(db_path),
    "swatch_snapshot": str(settings.swatch_snapshot_path),
    "swatch_count": len(dataset.swatches),
}


## Plan

- Hypothesis: deterministic outputs are now easy to inspect and judge without model-side ambiguity.
- Inputs to sample: exact swatch hits, muted warm tones, neutrals, and high-chroma colours.
- Readouts: recent rows, verdict counts, pending rows, and neutral rows worth tightening later.


In [ ]:
sample_hexes = (
    "#f3ece0",  # pale neutral
    "#d22345",  # loud red
    "#947764",  # muted warm tone
    "#0095c8",  # blue
)

def log_hexes(hex_values: tuple[str, ...]) -> list[int]:
    output_ids: list[int] = []
    for value in hex_values:
        state = build_one_up_state(value, dataset)
        output_ids.append(record_one_up_state(settings.eval_db_path, state))
    return output_ids

sample_hexes


In [ ]:
WRITE_SAMPLE_ROWS = False

if WRITE_SAMPLE_ROWS:
    logged_ids = log_hexes(sample_hexes)
else:
    logged_ids = []

{"write_sample_rows": WRITE_SAMPLE_ROWS, "logged_ids": logged_ids}


## Results

- Recent rows show the current deterministic output surface.
- Verdict counts show how much of the queue is still pending.
- Pending and neutral rows are the first place to look for taxonomy awkwardness.


In [ ]:
recent_rows = [dict(row) for row in list_outputs(settings.eval_db_path, limit=10)]
recent_rows


In [ ]:
counts(settings.eval_db_path)


In [ ]:
pending_rows = [
    dict(row)
    for row in list_outputs(settings.eval_db_path, limit=15, verdict="pending")
]
pending_rows


In [ ]:
with connect(settings.eval_db_path) as conn:
    family_counts = [
        dict(row)
        for row in conn.execute(
            """
            SELECT family, COUNT(*) AS count
            FROM eval_outputs
            GROUP BY family
            ORDER BY count DESC, family ASC
            """
        ).fetchall()
    ]

family_counts


In [ ]:
with connect(settings.eval_db_path) as conn:
    neutral_rows = [
        dict(row)
        for row in conn.execute(
            """
            SELECT
                id,
                input_hex,
                nearest_swatch_name,
                replacement_shade_name,
                current_rank,
                replacement_rank,
                current_verdict,
                distance_cie76,
                created_at
            FROM eval_outputs
            WHERE family = 'neutral'
            ORDER BY id DESC
            LIMIT 15
            """
        ).fetchall()
    ]

neutral_rows


## Next steps

- judge pending rows from the CLI with `huemiliator eval-judge <id> <pass|fail> --note "..."`
- tighten the neutral and warm boundary if these rows keep looking mushy
- widen the notebook only after this local judgment lane proves useful
